# 09 — Ablation Studies

Train a new Transformer from scratch for each ablation, changing exactly one thing.  
Baseline: Experiment C (mixed Heston + GAN paths).

| ID | What Changes | Hypothesis |
|----|-------------|------------|
| A1 | Remove vix_slope and vix_6m_slope | VIX term structure shape is more informative than VIX level alone |
| A2 | Remove all macro features | Macro features help in medium/long dated options |
| A3 | Sequence length 20 instead of 60 | 60 days captures a full vol cycle; 20 is insufficient |
| A4 | Sequence length 120 instead of 60 | Beyond 60 days adds noise |
| A5 | 1 encoder block instead of 3 | Depth matters for composing momentum with vol regime |
| A6 | Remove vol_regime label | Explicit regime label helps beyond raw features |
| A7 | Heston only vs GAN only vs mixed | GAN helps most in high vol regimes |

In [ ]:
import sys, os
import numpy as np
import pandas as pd
import torch
sys.path.insert(0, os.path.abspath('..'))

from src.models.transformer import build_transformer, build_ablation_transformer
from src.training.train_transformer import train_transformer
from src.data.dataset import SimulatedPricingDataset
from src.evaluation.pricing_eval import evaluate_ablations, predict_transformer, compute_metrics

PROCESSED_DIR = os.path.join('..', 'data', 'processed')
CHECKPOINT_BASE = os.path.join('..', 'checkpoints', 'transformer')
TABLES_DIR = os.path.join('..', 'results', 'tables')

device = torch.device('cuda' if torch.cuda.is_available() else
                       'mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
# Load data
train_seqs = np.load(os.path.join(PROCESSED_DIR, 'train_sequences.npy'))

h_contracts = np.load(os.path.join(PROCESSED_DIR, 'heston_contracts.npy'))
h_labels = np.load(os.path.join(PROCESSED_DIR, 'heston_labels.npy'))
g_contracts = np.load(os.path.join(PROCESSED_DIR, 'gan_contracts.npy'))
g_labels = np.load(os.path.join(PROCESSED_DIR, 'gan_labels.npy'))

# Mixed dataset (baseline for ablations)
mixed_contracts = np.concatenate([h_contracts[:25000], g_contracts[:25000]], axis=0)
mixed_labels = np.concatenate([h_labels[:25000], g_labels[:25000]], axis=0)

# Test options
opts_test = pd.read_pickle(os.path.join(PROCESSED_DIR, 'opts_test.pkl'))
print(f'Test options: {len(opts_test)}')

In [ ]:
# Load baseline (Experiment C)
baseline_path = os.path.join(CHECKPOINT_BASE, 'experiment_c', 'best_model.pt')
baseline_model = build_transformer()
baseline_model.load_state_dict(torch.load(baseline_path, map_location='cpu'))
baseline_model.eval()
print('Loaded Experiment C baseline.')

## Helper: Prepare Ablation Datasets

Some ablations modify the input features or sequence length.

In [ ]:
from src.data.preprocess import FEATURE_COLS

def make_ablation_sequences(ablation_id, train_seqs, feature_cols=FEATURE_COLS):
    """
    Modify training sequences for ablations that change input features or seq length.
    Returns modified sequences and the new feature count.
    """
    seqs = train_seqs.copy()
    
    if ablation_id == 'A1':
        # Remove vix_slope (idx 6) and vix_6m_slope (idx 7)
        keep = [i for i in range(seqs.shape[2]) if i not in [6, 7]]
        return seqs[:, :, keep], len(keep)
    
    elif ablation_id == 'A2':
        # Remove treasury_2y (16), put_call_ratio (17), dxy (18), credit_spread (14)
        remove_idx = [14, 16, 17, 18]
        keep = [i for i in range(seqs.shape[2]) if i not in remove_idx]
        return seqs[:, :, keep], len(keep)
    
    elif ablation_id == 'A3':
        # Sequence length 20
        return seqs[:, -20:, :], seqs.shape[2]
    
    elif ablation_id == 'A4':
        # Sequence length 120 — need to rebuild from scaled data
        # For now, pad with zeros at the start (or rebuild from train_scaled)
        train_scaled = np.load(os.path.join(PROCESSED_DIR, 'train_scaled.npy'))
        new_seqs = []
        for i in range(len(train_scaled) - 120):
            new_seqs.append(train_scaled[i:i+120])
        return np.array(new_seqs, dtype=np.float32), seqs.shape[2]
    
    elif ablation_id == 'A6':
        # Remove vol_regime (idx 19)
        keep = [i for i in range(seqs.shape[2]) if i != 19]
        return seqs[:, :, keep], len(keep)
    
    else:
        # A5 and A7 don't change the data, only the model
        return seqs, seqs.shape[2]

print('Ablation data helper ready.')

## Run Ablations

In [ ]:
ablation_models = {}
ablation_histories = {}

common_kwargs = dict(
    batch_size=256, max_epochs=100, lr=1e-4,
    patience=10, lr_patience=5, lr_factor=0.5,
    device=device,
)

for abl_id in ['A1', 'A2', 'A3', 'A4', 'A5', 'A6']:
    print(f'\n{"=" * 60}')
    print(f'Ablation {abl_id}')
    print(f'{"=" * 60}')
    
    abl_seqs, n_features = make_ablation_sequences(abl_id, train_seqs)
    seq_len = abl_seqs.shape[1]
    
    # Build dataset with modified sequences
    ds = SimulatedPricingDataset(abl_seqs, mixed_contracts, mixed_labels)
    
    # Build model
    model = build_ablation_transformer(abl_id, seq_len=seq_len)
    
    model, history = train_transformer(
        model, ds,
        checkpoint_dir=os.path.join(CHECKPOINT_BASE, f'ablation_{abl_id.lower()}'),
        experiment_name=f'Ablation {abl_id}',
        **common_kwargs,
    )
    
    ablation_models[abl_id] = model
    ablation_histories[abl_id] = history

## Ablation A7: Heston vs GAN vs Mixed

This is essentially comparing Experiments A, B, C — already done in notebook 08.

In [ ]:
print('Ablation A7 is the comparison of Experiments A, B, C.')
print('Results are already computed in 08_evaluate_pricing.')
print('See results/tables/overall_metrics.csv for the comparison.')

## Evaluate All Ablations

In [ ]:
# Note: For ablations A1, A2, A6 the test data also needs feature removal.
# For A3, A4 the test sequences need length adjustment.
# The evaluate_ablations function uses the standard opts_test which expects (60, 24) input.
# We need custom evaluation for feature-modified ablations.

# For now, evaluate A5 (only architecture change) directly:
from src.evaluation.pricing_eval import predict_transformer, compute_metrics

y_true = opts_test['mid_price'].values

# Baseline
baseline_preds = predict_transformer(baseline_model, opts_test, device=device)
baseline_mae = compute_metrics(y_true, baseline_preds)['MAE']
print(f'Baseline (C) MAE: {baseline_mae:.4f}')

rows = [{'Ablation': 'Baseline (C)', 'MAE': baseline_mae}]

# A5 uses same input format, just 1 encoder block
if 'A5' in ablation_models:
    preds_a5 = predict_transformer(ablation_models['A5'], opts_test, device=device)
    mae_a5 = compute_metrics(y_true, preds_a5)['MAE']
    rows.append({'Ablation': 'A5 (1 block)', 'MAE': mae_a5, 'Delta': mae_a5 - baseline_mae})
    print(f'A5 MAE: {mae_a5:.4f} (delta: {mae_a5 - baseline_mae:+.4f})')

# For other ablations, we note they need custom test evaluation
print('\nNote: Ablations A1-A4, A6 require modified test inputs.')
print('See discussion below for how to evaluate them properly.')

results_df = pd.DataFrame(rows)
print('\n', results_df.to_string(index=False))

## Evaluating Feature-Modified Ablations

For ablations that change input features (A1, A2, A6) or sequence length (A3, A4),
we need to modify the test data to match the training configuration.

In [ ]:
from src.data.dataset import OptionsDataset
from torch.utils.data import DataLoader

def predict_ablation(model, opts_df, ablation_id, device):
    """Custom prediction for ablations with modified inputs."""
    model = model.to(device).eval()
    preds = []
    
    for _, row in opts_df.iterrows():
        seq = np.array(row['sequence'], dtype=np.float32)  # (60, 20)
        contract = np.array([row['strike'], row['time_to_expiry'],
                             row['rate_input'], row['moneyness_input']], dtype=np.float32)
        
        # Modify sequence based on ablation
        if ablation_id == 'A1':
            seq = np.delete(seq, [6, 7], axis=1)  # Remove vix_slope, vix_6m_slope
        elif ablation_id == 'A2':
            seq = np.delete(seq, [14, 16, 17, 18], axis=1)  # Remove macro features
        elif ablation_id == 'A3':
            seq = seq[-20:]  # Last 20 days only
        elif ablation_id == 'A4':
            # Pad with zeros at start to get 120 steps
            pad = np.zeros((60, seq.shape[1]), dtype=np.float32)
            seq = np.concatenate([pad, seq], axis=0)  # 120 steps
        elif ablation_id == 'A6':
            seq = np.delete(seq, [19], axis=1)  # Remove vol_regime
        
        seq_len = seq.shape[0]
        contract_tiled = np.tile(contract, (seq_len, 1))
        full_input = np.concatenate([seq, contract_tiled], axis=-1)
        
        inp = torch.tensor(full_input, dtype=torch.float32).unsqueeze(0).to(device)
        with torch.no_grad():
            pred = model(inp).cpu().numpy().flatten()[0]
        preds.append(pred)
    
    return np.array(preds)

# Evaluate all ablations
full_results = [{'Ablation': 'Baseline (C)', 'MAE': baseline_mae}]

for abl_id in ['A1', 'A2', 'A3', 'A4', 'A5', 'A6']:
    if abl_id not in ablation_models:
        continue
    print(f'Evaluating {abl_id}...')
    
    if abl_id == 'A5':
        preds = predict_transformer(ablation_models[abl_id], opts_test, device=device)
    else:
        preds = predict_ablation(ablation_models[abl_id], opts_test, abl_id, device)
    
    metrics = compute_metrics(y_true, preds)
    full_results.append({
        'Ablation': abl_id, 
        'MAE': metrics['MAE'],
        'MSE': metrics['MSE'],
        'MAPE': metrics['MAPE'],
        'Delta_MAE': metrics['MAE'] - baseline_mae
    })
    print(f'  {abl_id} MAE: {metrics["MAE"]:.4f} (delta: {metrics["MAE"] - baseline_mae:+.4f})')

ablation_df = pd.DataFrame(full_results)
ablation_df.to_csv(os.path.join(TABLES_DIR, 'ablation_results.csv'), index=False)
print('\n' + ablation_df.to_string(index=False))

## Hypothesis Discussion

Fill in after running experiments:

In [ ]:
hypotheses = {
    'A1': 'VIX term structure shape is more informative than VIX level alone, especially in stress regimes.',
    'A2': 'Macro features help in medium/long dated options but less so in short dated.',
    'A3': '60 days captures a full vol cycle; 20 days is insufficient for regime detection.',
    'A4': 'Beyond 60 days adds noise rather than signal for short-dated options.',
    'A5': 'Depth matters because pricing requires composing short-term momentum with longer-term vol regime.',
    'A6': 'Explicit regime label helps over and above what the raw features encode.',
    'A7': 'GAN paths help most in high vol regimes where fat tails matter.',
}

print('Ablation Hypotheses vs Results')
print('=' * 80)
for abl_id, hyp in hypotheses.items():
    result_row = ablation_df[ablation_df['Ablation'] == abl_id]
    delta = result_row['Delta_MAE'].values[0] if len(result_row) else 'N/A'
    print(f'\n{abl_id}: {hyp}')
    if delta != 'N/A':
        direction = 'CONFIRMED' if delta > 0 else 'REFUTED'
        print(f'  Result: MAE delta = {delta:+.4f} -> Hypothesis {direction}')
        print(f'  (Positive delta = removing feature hurt performance = feature was useful)')
    else:
        print(f'  Result: Not evaluated')